# VisTacFusion v3 — Colab Training (Sim + Real Co-Training)

Single-GPU training on Colab (A100/V100/T4).

**Prerequisites on Google Drive:**
- `MyDrive/HDR_Lab/sim_data/*.tar` — per-object sim data tars
- `MyDrive/HDR_Lab/real_data/*.zip` — per-object real data zips
- `MyDrive/HDR_Lab/DINOv3_DPT/dinov3_vitl16_pretrain_lvd1689m.pth` — encoder weights
- `MyDrive/HDR_Lab/VisTacFusion/meshes.tar` — mesh .obj files (for pose label computation)

**Modes:** Set `TRAINING_MODE = 'sim'` for sim-only, or `'sim+real'` for co-training.

## 1. Setup: Mount Drive & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo
!git clone https://github.com/cynthiahuang1004/VisTacFusion.git /content/VisTacFusion
%cd /content/VisTacFusion
!git checkout VisTacFusion-v2

In [ ]:
# Install dependencies
!pip install -e . -q
!pip install trimesh scipy -q

## 2. Training Mode & Data Extraction

Set `TRAINING_MODE` to control sim-only vs co-training.
Extract all data to `/content/data/` (local disk is faster than Drive).

In [ ]:
import os, glob, time

# ---- Training mode ----
TRAINING_MODE = 'sim+real'  # 'sim' or 'sim+real'

# ---- Drive paths ----
DRIVE_SIM_DATA  = '/content/drive/MyDrive/HDR_Lab/sim_data'
DRIVE_REAL_DATA = '/content/drive/MyDrive/HDR_Lab/real_data'
DRIVE_MESHES    = '/content/drive/MyDrive/HDR_Lab/VisTacFusion/meshes.tar'
DRIVE_WEIGHTS   = '/content/drive/MyDrive/HDR_Lab/DINOv3_DPT/dinov3_vitl16_pretrain_lvd1689m.pth'

# ---- Local paths ----
LOCAL_RENDERS   = '/content/data/renders'
LOCAL_REAL      = '/content/data/real_data'
LOCAL_MESHES    = '/content/data/meshes'
LOCAL_WEIGHTS   = '/content/VisTacFusion/weights/dinov3_vitl16_pretrain_lvd1689m.pth'

os.makedirs(LOCAL_RENDERS, exist_ok=True)
os.makedirs(LOCAL_REAL, exist_ok=True)
os.makedirs(os.path.dirname(LOCAL_WEIGHTS), exist_ok=True)

print(f'Training mode: {TRAINING_MODE}')

In [ ]:
# Extract meshes
if not os.path.exists(LOCAL_MESHES):
    print('Extracting meshes...')
    !tar xf "{DRIVE_MESHES}" -C /content/data/
    print(f'  -> {len(os.listdir(LOCAL_MESHES))} files')
else:
    print('Meshes already extracted')

In [ ]:
# Extract all sim object tars
tars = sorted(glob.glob(os.path.join(DRIVE_SIM_DATA, '*.tar')))
print(f'Found {len(tars)} sim object tars on Drive')

for i, tar_path in enumerate(tars):
    obj_name = os.path.splitext(os.path.basename(tar_path))[0]
    obj_dir = os.path.join(LOCAL_RENDERS, obj_name)
    if os.path.exists(obj_dir):
        print(f'  [{i+1}/{len(tars)}] {obj_name} — already extracted')
        continue
    t0 = time.time()
    !tar xf "{tar_path}" -C "{LOCAL_RENDERS}/"
    elapsed = time.time() - t0
    n_sessions = len([d for d in os.listdir(obj_dir) if d.startswith('session')]) if os.path.exists(obj_dir) else 0
    print(f'  [{i+1}/{len(tars)}] {obj_name} — {n_sessions} sessions ({elapsed:.0f}s)')

In [ ]:
# Extract real data zips (only in sim+real mode)
if TRAINING_MODE == 'sim+real':
    zips = sorted(glob.glob(os.path.join(DRIVE_REAL_DATA, '*.zip')))
    print(f'Found {len(zips)} real data zips on Drive')

    for i, zip_path in enumerate(zips):
        obj_name = os.path.splitext(os.path.basename(zip_path))[0]
        obj_dir = os.path.join(LOCAL_REAL, obj_name)
        if os.path.exists(obj_dir):
            print(f'  [{i+1}/{len(zips)}] {obj_name} — already extracted')
            continue
        t0 = time.time()
        !unzip -q "{zip_path}" -d "{LOCAL_REAL}/"
        elapsed = time.time() - t0
        n_samples = len(glob.glob(os.path.join(obj_dir, '**/samples/*.png'), recursive=True))
        print(f'  [{i+1}/{len(zips)}] {obj_name} — {n_samples} samples ({elapsed:.0f}s)')
else:
    print('Sim-only mode — skipping real data')

In [ ]:
# Copy DINOv3 weights
if not os.path.exists(LOCAL_WEIGHTS):
    print('Copying DINOv3 weights...')
    !cp "{DRIVE_WEIGHTS}" "{LOCAL_WEIGHTS}"
    print(f'  -> {os.path.getsize(LOCAL_WEIGHTS) / 1e9:.1f} GB')
else:
    print('Weights already copied')

In [ ]:
# Verify data
objects = sorted(d for d in os.listdir(LOCAL_RENDERS) if os.path.isdir(os.path.join(LOCAL_RENDERS, d)))
total_sessions = sum(
    len([s for s in os.listdir(os.path.join(LOCAL_RENDERS, o)) if s.startswith('session')])
    for o in objects
)
print(f'Sim: {len(objects)} objects, {total_sessions} total sessions')
print(f'  Objects: {objects}')

if TRAINING_MODE == 'sim+real' and os.path.exists(LOCAL_REAL):
    real_objects = sorted(d for d in os.listdir(LOCAL_REAL) if os.path.isdir(os.path.join(LOCAL_REAL, d)))
    real_samples = sum(
        len(glob.glob(os.path.join(LOCAL_REAL, o, '**/samples/*.png'), recursive=True))
        for o in real_objects
    )
    print(f'Real: {len(real_objects)} objects, {real_samples} total samples')
    print(f'  Objects: {real_objects}')

!du -sh /content/data/

## 3. Configure & Build Model

In [ ]:
import yaml

# --- data.yaml overrides ---
with open('configs/data.yaml') as f:
    data_cfg = yaml.safe_load(f)

data_cfg['dataset'] = TRAINING_MODE
data_cfg['sim']['root'] = LOCAL_RENDERS
data_cfg['sim']['mesh_dir'] = LOCAL_MESHES
data_cfg['loader']['num_workers'] = 4
data_cfg['loader']['persistent_workers'] = False

if TRAINING_MODE == 'sim+real':
    data_cfg['real']['root'] = LOCAL_REAL
    data_cfg['real']['mesh_dir'] = LOCAL_MESHES

with open('configs/data.yaml', 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False, sort_keys=False)

# --- model.yaml overrides (object embedding for co-training) ---
with open('configs/model.yaml') as f:
    model_cfg = yaml.safe_load(f)

if TRAINING_MODE == 'sim+real':
    model_cfg['tokens']['object_embedding'] = True
else:
    model_cfg['tokens']['object_embedding'] = False

with open('configs/model.yaml', 'w') as f:
    yaml.dump(model_cfg, f, default_flow_style=False, sort_keys=False)

# --- train.yaml overrides ---
with open('configs/train.yaml') as f:
    train_cfg = yaml.safe_load(f)

train_cfg['batch_size'] = 64
train_cfg['num_workers'] = 4

with open('configs/train.yaml', 'w') as f:
    yaml.dump(train_cfg, f, default_flow_style=False, sort_keys=False)

print(f'Configs updated for Colab ({TRAINING_MODE}).')
print(f"  dataset: {data_cfg['dataset']}")
print(f"  batch_size: {train_cfg['batch_size']}")
print(f"  sim.root: {data_cfg['sim']['root']}")
if TRAINING_MODE == 'sim+real':
    print(f"  real.root: {data_cfg['real']['root']}")
    print(f"  object_embedding: {model_cfg['tokens']['object_embedding']}")

In [ ]:
# Verify model builds and dataset loads
from vistacfusion.utils.config import merge_configs
from vistacfusion.models.model import build_model
from vistacfusion.data.dataset import build_datasets
from vistacfusion.utils.misc import param_count_str

cfg = merge_configs('configs/model.yaml', 'configs/train.yaml', 'configs/data.yaml')

train_ds, val_ds = build_datasets(cfg)
print(f'Train: {len(train_ds)} samples | Val: {len(val_ds)} samples')

model = build_model(cfg)
print(f'Model: {param_count_str(model)}')

In [ ]:
# Quick shape check
import torch

device = torch.device('cuda')
model = model.to(device)

sample = train_ds[0]
rgb = sample['rgb'].unsqueeze(0).to(device)
tac = sample['tactile'].unsqueeze(0).to(device)

with torch.no_grad(), torch.autocast('cuda'):
    out = model(rgb, tac, config='both')

print('Output shapes:')
for k, v in out.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k}: {v.shape}')

del model, rgb, tac, out
torch.cuda.empty_cache()

## 4. Train

In [ ]:
import time, os, json, math, random
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from vistacfusion.utils.config import merge_configs
from vistacfusion.utils.misc import set_seed, param_count_str
from vistacfusion.models.model import build_model
from vistacfusion.data.dataset import build_datasets
from vistacfusion.losses.total import MultiTaskLoss
from vistacfusion.engine.eval import evaluate, precompute_encoder_cache
from vistacfusion.engine.train import (
    build_optimizer, build_scheduler, save_checkpoint, load_checkpoint,
    sample_config, sample_dpt_inject, save_loss_plots
)

# ---- Config ----
cfg = merge_configs('configs/model.yaml', 'configs/train.yaml', 'configs/data.yaml')
set_seed(cfg.seed)

device = torch.device('cuda')

# Output dir on Drive (persists across sessions)
OUTPUT_DIR = '/content/drive/MyDrive/HDR_Lab/VisTacFusion/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Optional: resume from a previous checkpoint
RESUME_FROM = None  # e.g. f'{OUTPUT_DIR}/epoch_029.pt'

print(f'Output: {OUTPUT_DIR}')
print(f'Resume: {RESUME_FROM}')

In [ ]:
# ---- Data ----
train_ds, val_ds = build_datasets(cfg)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

lc = cfg.loader
train_loader = DataLoader(
    train_ds, batch_size=cfg.batch_size, shuffle=True,
    num_workers=lc.num_workers, pin_memory=lc.pin_memory,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.batch_size, shuffle=False,
    num_workers=lc.num_workers, pin_memory=lc.pin_memory,
)
print(f'Steps/epoch: {len(train_loader)}')

In [ ]:
# ---- Model + Loss + Optimizer ----
model = build_model(cfg).to(device)
print(f'Model: {param_count_str(model)}')

criterion = MultiTaskLoss(
    cfg.loss, pose_mode=cfg.heads.pose.pose_mode,
    rot_num_bins=cfg.heads.pose.get('rot_num_bins', 72),
).to(device)

optimizer = build_optimizer(model, cfg.optim, criterion)
scheduler = build_scheduler(optimizer, cfg, len(train_loader))
scaler = torch.amp.GradScaler('cuda', enabled=cfg.amp)

start_epoch = 0
best_metric = float('inf')
best_pose_metric = float('inf')

if RESUME_FROM:
    print(f'Resuming from {RESUME_FROM}')
    start_epoch, best_metric = load_checkpoint(
        RESUME_FROM, model, optimizer, scheduler, scaler,
        criterion=criterion, device=device)
    start_epoch += 1
    print(f'  -> epoch {start_epoch}, best_metric={best_metric:.4f}')

In [ ]:
# ---- Val encoder cache ----
print('Pre-computing val encoder cache...')
val_enc_cache = precompute_encoder_cache(model, val_loader, device)

In [ ]:
# ---- Training loop ----
max_epochs = cfg.schedule.max_epochs
md_cfg = cfg.modality_dropout
writer = SummaryWriter(log_dir=os.path.join(OUTPUT_DIR, 'tb'))
history_path = os.path.join(OUTPUT_DIR, 'history.json')
plot_dir = os.path.join(OUTPUT_DIR, 'plots')
os.makedirs(plot_dir, exist_ok=True)

history = []
if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)

print(f'Training epochs {start_epoch} -> {max_epochs}')
print(f'batch_size={cfg.batch_size}, steps/epoch={len(train_loader)}')
print(f'mode={cfg.dataset}')
print('=' * 80)

for epoch in range(start_epoch, max_epochs):
    model.train()
    running = {}
    t0 = time.time()

    for step, batch in enumerate(train_loader):
        batch = {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v)
                 for k, v in batch.items()}

        config = sample_config(md_cfg)
        inject_rgb = sample_dpt_inject(md_cfg)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', enabled=cfg.amp):
            obj_ids = batch.get('object')
            out = model(batch['rgb'], batch['tactile'], config=config,
                        inject_rgb_to_dpt=inject_rgb, object_ids=obj_ids)
            gt = {'depth': batch['depth'], 'normal': batch['normal'],
                  'pose': batch['pose'], 'mask': batch.get('mask')}
            loss, comps = criterion(out, gt, supervise_dense=True)

        if torch.isnan(loss) or torch.isinf(loss):
            print(f'  [WARN] NaN/Inf loss at epoch {epoch} step {step}, skipping')
            optimizer.zero_grad(set_to_none=True)
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        for k, v in comps.items():
            running[k] = running.get(k, 0.0) + float(v)

        if step % cfg.log_every == 0:
            lr_now = optimizer.param_groups[0]['lr']
            msg = '  '.join(f'{k}={running[k]/(step+1):.4f}' for k in sorted(running))
            inj = '+rgb' if inject_rgb else ''
            print(f'[epoch {epoch:03d} | step {step:04d}/{len(train_loader)} | '
                  f'cfg={config:7s}{inj} | lr={lr_now:.2e}] {msg}')

    elapsed = time.time() - t0
    train_metrics = {k: v / len(train_loader) for k, v in running.items()}
    print(f'[epoch {epoch:03d}] train done in {elapsed:.0f}s  '
          f'avg_total={train_metrics.get("total", 0):.4f}')

    # ---- Validation ----
    val_metrics = evaluate(model, val_loader, cfg, device, encoder_cache=val_enc_cache)
    print(f'[epoch {epoch:03d}] val metrics: {val_metrics}')

    for config_name, metrics in val_metrics.items():
        for mk, mv in metrics.items():
            writer.add_scalar(f'val_{config_name}/{mk}', mv, epoch)
    for k, v in train_metrics.items():
        writer.add_scalar(f'train_epoch/{k}', v, epoch)

    history.append({'epoch': epoch, 'train': train_metrics, 'val': val_metrics})
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)
    save_loss_plots(history, plot_dir)

    # ---- Checkpoints ----
    both_metrics = val_metrics.get('both', {})
    depth_score = both_metrics.get('depth_mse', float('inf'))
    pose_score = both_metrics.get('pose_rot', float('inf'))

    if depth_score < best_metric:
        best_metric = depth_score
        save_checkpoint(os.path.join(OUTPUT_DIR, 'best_depth.pt'),
                        model, optimizer, scheduler, scaler, epoch, best_metric,
                        criterion=criterion)
        print(f'  ** new best depth: mse={best_metric:.6f}')

    if pose_score < best_pose_metric:
        best_pose_metric = pose_score
        save_checkpoint(os.path.join(OUTPUT_DIR, 'best_pose.pt'),
                        model, optimizer, scheduler, scaler, epoch, best_pose_metric,
                        criterion=criterion)
        print(f'  ** new best pose: rot={best_pose_metric:.4f}')

    if epoch % cfg.ckpt_every_epochs == 0 or epoch == max_epochs - 1:
        save_checkpoint(os.path.join(OUTPUT_DIR, f'epoch_{epoch:03d}.pt'),
                        model, optimizer, scheduler, scaler, epoch, best_metric,
                        criterion=criterion)

    writer.flush()

print('Training complete.')
writer.close()

## 5. Evaluation

In [ ]:
# Load best checkpoint and evaluate
best_ckpt = os.path.join(OUTPUT_DIR, 'best_depth.pt')
if os.path.exists(best_ckpt):
    model_eval = build_model(cfg).to(device)
    load_checkpoint(best_ckpt, model_eval, device=device)
    print('Loaded best_depth.pt')

    val_metrics = evaluate(model_eval, val_loader, cfg, device,
                           encoder_cache=val_enc_cache)
    print('\n=== Best Depth Checkpoint ===')
    for config_name, metrics in val_metrics.items():
        print(f'\n  {config_name}:')
        for k, v in metrics.items():
            print(f'    {k}: {v}')
else:
    print('No best checkpoint found. Train first.')

In [ ]:
# Show training curves
from IPython.display import Image as IPImage, display
import glob as _glob

for png in sorted(_glob.glob(os.path.join(plot_dir, '*.png'))):
    print(os.path.basename(png))
    display(IPImage(filename=png, width=800))